# Hourly Energy Consumption Time Series Forecasting

This notebook demonstrates an end-to-end machine learning workflow for power grid demand forecasting. It covers datetime indexing, calendar and lag feature engineering, chronological train/test splitting, gradient boosted decision tree regression (XGBoost), and residual diagnostic evaluation.

## 1. Environment & Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"


## 2. Data Ingestion & Time Indexing

Load hourly energy consumption records and set the Datetime timestamp as the DataFrame index.

In [ ]:
df = pd.read_csv("dataset/energy_consumption.csv")
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.set_index("Datetime")

print(f"Dataset shape: {df.shape}")
df.head()


## 3. Exploratory Data Analysis & Seasonality Patterns

Visualize overall multi-year demand timeline and inspect daily and annual seasonality profiles.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Daily seasonality
df_daily = df.copy()
df_daily["hour"] = df_daily.index.hour
sns.boxplot(data=df_daily, x="hour", y="EnergyDemand_MW", ax=ax1, color="#457b9d")
ax1.set_title("Hourly Demand Profile (Daily Seasonality)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Hour of Day (0-23)")
ax1.set_ylabel("Demand (MW)")

# Monthly seasonality
df_monthly = df.copy()
df_monthly["month"] = df_monthly.index.month
sns.boxplot(data=df_monthly, x="month", y="EnergyDemand_MW", ax=ax2, color="#e76f51")
ax2.set_title("Monthly Demand Distribution (Annual Seasonality)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Month (1-12)")
ax2.set_ylabel("Demand (MW)")

plt.tight_layout()
plt.show()


## 4. Feature Engineering (Calendar, Lag, & Rolling Windows)

Extract temporal features from timestamps, generate autoregressive lag values, and compute rolling statistics.

In [ ]:
# Calendar features
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["quarter"] = df.index.quarter
df["month"] = df.index.month
df["year"] = df.index.year
df["dayofyear"] = df.index.dayofyear
df["is_weekend"] = np.where(df["dayofweek"] >= 5, 1, 0)

# Autoregressive lag features
df["lag_1"] = df["EnergyDemand_MW"].shift(1)
df["lag_24"] = df["EnergyDemand_MW"].shift(24)
df["lag_168"] = df["EnergyDemand_MW"].shift(168)

# Rolling window statistics
df["rolling_mean_24"] = df["EnergyDemand_MW"].shift(1).rolling(24).mean()
df["rolling_std_24"] = df["EnergyDemand_MW"].shift(1).rolling(24).std()

# Drop rows with NaN caused by initial lags
df = df.dropna()

features = [
    "hour", "dayofweek", "quarter", "month", "year", "dayofyear", "is_weekend",
    "lag_1", "lag_24", "lag_168", "rolling_mean_24", "rolling_std_24"
]
target = "EnergyDemand_MW"

print(f"Engineered feature count: {len(features)}")
df[features].head()


## 5. Chronological Train / Test Partitioning

Time series models require strict sequential splits to prevent future lookahead leakage. We use all data up to July 31, 2023 for training, and evaluate on August 1, 2023 onwards.

In [ ]:
split_date = "2023-08-01"
train = df.loc[df.index < split_date]
test = df.loc[df.index >= split_date]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print(f"Training observations: {len(X_train)}")
print(f"Testing observations:  {len(X_test)}")


## 6. Model Training: XGBoost Regressor

Train a gradient boosted decision tree regressor with early stopping to prevent overfitting.

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

print("XGBoost training completed successfully.")


## 7. Model Evaluation & Performance Metrics

Evaluate predictions on the out-of-time test set using RMSE, MAE, MAPE, and R2 score.

In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = r2_score(y_test, y_pred)

print(f"Root Mean Squared Error (RMSE): {rmse:.2f} MW")
print(f"Mean Absolute Error (MAE):       {mae:.2f} MW")
print(f"Mean Absolute Percentage Error:  {mape:.2f}%")
print(f"R-squared Score (R2):            {r2:.4f}")


## 8. Feature Importance Analysis

Identify which engineered predictors contributed most to the model's split decisions.

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importance.plot(kind="barh", color="#2a9d8f")
plt.title("XGBoost Feature Importance", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("F-Score / Relative Weight")
plt.ylabel("Features")
plt.tight_layout()
plt.show()


## 9. Forecast vs Actual Visualization

Compare model predictions against ground truth demand over a 2-week test sample horizon.

In [ ]:
sample_window = test.loc["2023-10-01":"2023-10-15"]
sample_pred = pd.Series(model.predict(sample_window[features]), index=sample_window.index)

plt.figure(figsize=(14, 5))
plt.plot(sample_window.index, sample_window[target], label="Actual Demand", color="#1d3557", lw=1.8)
plt.plot(sample_window.index, sample_pred, label="XGBoost Forecast", color="#e63946", linestyle="--", lw=1.8)
plt.fill_between(sample_window.index, sample_pred - rmse, sample_pred + rmse, color="#e63946", alpha=0.15, label="RMSE Error Band")
plt.title("Actual vs Predicted Demand (Oct 1 - Oct 15, 2023)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Datetime")
plt.ylabel("Demand (MW)")
plt.legend(frameon=True, facecolor="white")
plt.tight_layout()
plt.show()


## 10. Residual Diagnostics & Model Serialization

Verify error normality and serialize trained weights for downstream deployment.

In [ ]:
residuals = y_test - y_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(y_pred, residuals, alpha=0.3, color="#457b9d", s=12)
ax1.axhline(0, color="red", linestyle="--", lw=1.5)
ax1.set_title("Residuals vs Predicted Values", fontsize=12, fontweight="bold")
ax1.set_xlabel("Predicted Demand (MW)")
ax1.set_ylabel("Residual Error (MW)")

sns.histplot(residuals, kde=True, ax=ax2, color="#1d3557", bins=40)
ax2.axvline(0, color="red", linestyle="--", lw=1.5)
ax2.set_title("Residual Distribution (Gaussian Fit)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Residual Error (MW)")
ax2.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# Serialize model
import os
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/xgboost_energy_model.joblib")
print("Model successfully exported to models/xgboost_energy_model.joblib")
